# Sydney House Price Prediction Using Machine Learning

## Project overview

This project predicts Sydney property prices using machine learning. The focus is not only model accuracy, but also building a clean and reliable machine learning workflow using:

- train/test split
- preprocessing pipelines
- missing value handling
- categorical encoding
- log transformation of the target
- outlier handling
- model evaluation using MAE and RMSE
- feature importance interpretation

## Problem type

This is a **supervised regression problem** because the target variable is a continuous numeric value: property price.


## Business problem

Property buyers, real estate analysts, and financial institutions often need fast estimates of property values. This project builds a baseline machine learning model to estimate Sydney house prices using property features such as bedrooms, bathrooms, parking, property size, suburb information, distance from CBD, cash rate, and property inflation index.

The final model is useful as a portfolio learning project and analytical baseline. It is **not production-ready** because the prediction error is still high for real-world valuation decisions.


## Dataset

Dataset used: `alexlau203/sydney-house-prices` from Kaggle.

The dataset contains Sydney property sales information, including property characteristics, suburb-level statistics, and market-related features.

> Note: You need Kaggle/KaggleHub access to download the dataset.


In [1]:
# Optional: install kagglehub if needed
# !pip install kagglehub

In [2]:
import os
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

## 1. Load the data

In [3]:
path = kagglehub.dataset_download("alexlau203/sydney-house-prices")
csv_path = os.path.join(path, "domain_properties.csv")

housing_data = pd.read_csv(csv_path)

print("Dataset shape:", housing_data.shape)
housing_data.head()

Dataset shape: (11160, 17)


,price,date_sold,suburb,num_bath,num_bed,num_parking,property_size,type,suburb_population,suburb_median_income,suburb_sqkm,suburb_lat,suburb_lng,suburb_elevation,cash_rate,property_inflation_index,km_from_cbd
0,530000,13/1/16,Kincumber,4,4,2,1351,House,7093,29432,9.914,-33.47252,151.40208,24,2.0,150.9,47.05
1,525000,13/1/16,Halekulani,2,4,2,594,House,2538,24752,1.397,-33.21772,151.55237,23,2.0,150.9,78.54
2,480000,13/1/16,Chittaway Bay,2,4,2,468,House,2028,31668,1.116,-33.32678,151.44557,3,2.0,150.9,63.59
3,452000,13/1/16,Leumeah,1,3,1,344,House,9835,32292,4.055,-34.05375,150.83957,81,2.0,150.9,40.12
4,365500,13/1/16,North Avoca,0,0,0,1850,Vacant land,2200,45084,1.497,-33.45608,151.43598,18,2.0,150.9,49.98


## 2. Initial data inspection

In [4]:
housing_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11160 entries, 0 to 11159
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   price                     11160 non-null  int64  
 1   date_sold                 11160 non-null  object 
 2   suburb                    11160 non-null  object 
 3   num_bath                  11160 non-null  int64  
 4   num_bed                   11160 non-null  int64  
 5   num_parking               11160 non-null  int64  
 6   property_size             11160 non-null  int64  
 7   type                      11160 non-null  object 
 8   suburb_population         11160 non-null  int64  
 9   suburb_median_income      11160 non-null  int64  
 10  suburb_sqkm               11160 non-null  float64
 11  suburb_lat                11160 non-null  float64
 12  suburb_lng                11160 non-null  float64
 13  suburb_elevation          11160 non-null  int64  
 14  cash_r

In [5]:
housing_data.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
price,11160.0,NaN,NaN,NaN,1675395.267294,1290370.963076,225000.0,1002000.0,1388000.0,2020000.0,60000000.0
date_sold,11160,1338,12/11/21,121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
suburb,11160,637,Bateau Bay,68,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_bath,11160.0,NaN,NaN,NaN,2.073566,1.184881,0.0,1.0,2.0,3.0,46.0
num_bed,11160.0,NaN,NaN,NaN,3.758961,1.559743,0.0,3.0,4.0,4.0,47.0
num_parking,11160.0,NaN,NaN,NaN,2.017473,1.45456,0.0,1.0,2.0,2.0,50.0
property_size,11160.0,NaN,NaN,NaN,723.012366,1048.983662,7.0,430.0,600.0,765.0,59100.0
type,11160,16,House,9583,NaN,NaN,NaN,NaN,NaN,NaN,NaN
suburb_population,11160.0,NaN,NaN,NaN,9311.560036,7541.636246,22.0,3977.0,7457.0,12158.25,47176.0
suburb_median_income,11160.0,NaN,NaN,NaN,40168.243369,11089.95512,14248.0,32448.0,39104.0,45552.0,97500.0


## 3. Date feature engineering

The raw `date_sold` column is a string date. Machine learning models cannot directly use date strings, so the date is converted into separate numeric features:

- year
- month
- day

The original date column is then removed.


In [6]:
housing_data["date_sold"] = pd.to_datetime(
    housing_data["date_sold"],
    dayfirst=True,
    errors="coerce"
)

housing_data["year"] = housing_data["date_sold"].dt.year
housing_data["month"] = housing_data["date_sold"].dt.month
housing_data["day"] = housing_data["date_sold"].dt.day

housing_data = housing_data.drop("date_sold", axis=1)

housing_data[["year", "month", "day"]].isna().sum()

/tmp/ipykernel_16/4062556769.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  housing_data["date_sold"] = pd.to_datetime(


year     0
month    0
day      0
dtype: int64

## 4. Outlier handling

Sydney property prices can contain extreme luxury properties. These may represent a different market segment and can strongly affect RMSE.

For this beginner portfolio project, the top 1% most expensive properties are removed so the model focuses on the main residential market.


In [7]:
upper_limit = housing_data["price"].quantile(0.99)

housing_data_clean = housing_data[housing_data["price"] <= upper_limit].copy()

print("Original shape:", housing_data.shape)
print("Clean shape:", housing_data_clean.shape)
print("Rows removed:", housing_data.shape[0] - housing_data_clean.shape[0])
print("99th percentile price:", upper_limit)

Original shape: (11160, 19)
Clean shape: (11049, 19)
Rows removed: 111
99th percentile price: 6200000.0


In [8]:
housing_data_clean["price"].describe()

count    1.104900e+04
mean     1.601764e+06
std      9.123092e+05
min      2.250000e+05
25%      1.000000e+06
50%      1.375000e+06
75%      2.000000e+06
max      6.200000e+06
Name: price, dtype: float64

## 5. Feature and target split

The target variable `price` is log-transformed using `np.log1p`.

Why use log transformation?

Property prices are usually skewed. A small number of very expensive properties can dominate model errors. Log transformation compresses large values and helps the model learn a more stable pattern.

Later, predictions are converted back to dollars using `np.expm1`.


In [9]:
X = housing_data_clean.drop("price", axis=1)
y = np.log1p(housing_data_clean["price"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11049, 18)
y shape: (11049,)


## 6. Train-test split

The data is split before preprocessing to reduce data leakage.

Data leakage happens when information from the test set is accidentally used during training or preprocessing. The test set should stay unseen until final evaluation.


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 8839
Testing rows: 2210


## 7. Identify numeric and categorical features

In [11]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

Numeric Features: Index(['num_bath', 'num_bed', 'num_parking', 'property_size',
       'suburb_population', 'suburb_median_income', 'suburb_sqkm',
       'suburb_lat', 'suburb_lng', 'suburb_elevation', 'cash_rate',
       'property_inflation_index', 'km_from_cbd'],
      dtype='object')
Categorical Features: Index(['suburb', 'type'], dtype='object')


## 8. Build preprocessing pipeline

Numeric columns:
- missing values filled with median

Categorical columns:
- missing values filled with most frequent value
- categories converted to numeric columns using one-hot encoding

This is done inside a `ColumnTransformer` so preprocessing is clean and reusable.


In [12]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 9. Evaluation helper function

The model is trained on log prices, so predictions and true values are converted back to dollars before calculating MAE and RMSE.


In [13]:
def evaluate_log_model(model, X_test, y_test_log):
    """Evaluate a model trained on log-transformed target values."""
    y_pred_log = model.predict(X_test)

    y_test_dollars = np.expm1(y_test_log)
    y_pred_dollars = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_test_dollars, y_pred_dollars)
    rmse = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))

    return mae, rmse

## 10. Model training and comparison

Three models are compared:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor

Random Forest is expected to perform better because it combines many decision trees, which reduces overfitting and gives more stable predictions.


In [14]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    mae, rmse = evaluate_log_model(pipeline, X_test, y_test)

    results.append({
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse
    })

results_df = pd.DataFrame(results).sort_values(by="MAE")
results_df

,model,MAE,RMSE
2,Random Forest,270095.076649,460306.765804
0,Linear Regression,313994.019163,515481.333341
1,Decision Tree,386021.777466,657460.784938


## 11. Final model

The Random Forest pipeline is selected as the final model because it gave the best performance during experimentation.

Result from the latest run:

- MAE: approximately **$270,230**
- RMSE: approximately **$460,881**

This means the model is still making large errors and is not ready for production use. However, it is a strong beginner portfolio project because it demonstrates a proper machine learning workflow.


In [15]:
final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

final_model.fit(X_train, y_train)

final_mae, final_rmse = evaluate_log_model(final_model, X_test, y_test)

print("Final Random Forest Pipeline")
print("MAE:", final_mae)
print("RMSE:", final_rmse)

Final Random Forest Pipeline
MAE: 270095.07664854324
RMSE: 460306.76580439386


## 12. Feature importance

Feature importance helps explain what the Random Forest model is using to make predictions.

Because categorical columns are one-hot encoded inside the pipeline, the encoded feature names are extracted before matching them with Random Forest importance scores.


In [16]:
rf_model = final_model.named_steps["model"]

onehot = final_model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .named_steps["onehot"]

categorical_feature_names = onehot.get_feature_names_out(categorical_features)

all_feature_names = list(numeric_features) + list(categorical_feature_names)

feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance.head(20)

,feature,importance
12,km_from_cbd,0.270493
3,property_size,0.112856
8,suburb_lng,0.104727
1,num_bed,0.084923
10,cash_rate,0.084757
0,num_bath,0.082609
11,property_inflation_index,0.068899
5,suburb_median_income,0.049674
4,suburb_population,0.026086
7,suburb_lat,0.022786


## Feature importance interpretation

The strongest feature from experimentation was `km_from_cbd`, followed by property size, suburb longitude, number of bedrooms, cash rate, number of bathrooms, and property inflation index.

This makes sense because Sydney property prices are strongly affected by location, distance from CBD, property size, property type, and market conditions.

Individual suburb dummy variables may have low importance because the suburb feature is split across many one-hot encoded columns. The suburb effect is therefore spread across many separate features.


## Final conclusion

The final Random Forest pipeline achieved the best result in this project, with an MAE of approximately $270K and RMSE of approximately $461K.

The model learns realistic housing patterns, especially the importance of distance from CBD, property size, suburb location, bedrooms, bathrooms, and market conditions.

However, the error is still too high for real-world deployment. More work is needed before this could be used as a serious valuation tool.
